**FAMA FRENCH 3 FACTOR MODEL**

Okay, we've seen the basics of factor modeling and understood CAPM, now let's move on and do some more pratical things. FAMA French is a three factor model that expands the classic CAPM model. Eugene Fama and Kenneth French added two Risk Factors to the model: The size factor $SMB$ and the value factor $HML$.

Before we dig deeper, let's first understand these two factors.

$SMB$ stands for small minus big and stand for the outperformance of small cap stocks relative to large cap stocks. 

$HML$ stands for high minus low and explains the outperformance of value stocks with a high book to market ratio relative to grwoth stocks that typically consist of low book to market rates.

With these two factors we now get the *FAMA French formula*:

$$

\begin{aligned}
\\
\\
E(R_i) = R_f + \beta_(i_1) * (E(R_m)) - R_f) + \beta_(i_2) * SMB + \beta_(i_3) * HML
\\
\\
\end{aligned}
$$

Okay, now that we know how FAMA French works, we can move on to a real-life example on it's implementation.

$$Example\, of\, Fama\, French\, on\, AAPL$$

The factor data comes directly from [Kenneth French's Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) (*Fama/French 3 Factors*, daily). The file contains the daily returns of $R_m - R_f$ (`Mkt-RF`), $SMB$, $HML$ and the risk-free rate $R_f$ (`RF`), all in percent.

Unlike in the formula above, we regress on actual daily returns, so we estimate:

$$
R_i - R_f = \alpha_i + \beta_{i_1} \cdot (R_m - R_f) + \beta_{i_2} \cdot SMB + \beta_{i_3} \cdot HML + \varepsilon_i
$$

In [1]:
import io
import zipfile
import urllib.request

import pandas as pd
import yfinance as yf
import statsmodels.api as sm

# Download the Fama/French 3 factors (daily) from Kenneth French's data library
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"
with urllib.request.urlopen(url) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as z:
        raw = z.read(z.namelist()[0]).decode("utf-8")

ff = pd.read_csv(io.StringIO(raw), skiprows=4, index_col=0).dropna()  # dropna removes the copyright footer
ff.index = pd.to_datetime(ff.index.astype(str).str.strip(), format="%Y%m%d")
ff = ff.astype(float) / 100  # percent -> decimal

# Daily AAPL returns
df = yf.download("AAPL", period="max", interval="1d", auto_adjust=True)
returns = df["Close"]["AAPL"].pct_change().rename("AAPL")

data = ff.join(returns, how="inner").dropna()
data["AAPL-RF"] = data["AAPL"] - data["RF"]  # excess return of AAPL

X = data[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)  # -> This will be your Alpha
y = data["AAPL-RF"]

model = sm.OLS(y, X).fit()
print(model.summary())

alpha = model.params["const"]
print("\nFama French Results:")
print(f"Alpha (daily): {alpha:.5f}  ->  annualized: {alpha * 252:.2%}")
print(f"Market Beta: {model.params['Mkt-RF']:.2f}")
print(f"SMB Beta: {model.params['SMB']:.2f}")
print(f"HML Beta: {model.params['HML']:.2f}")
print(f"R-squared: {model.rsquared:.2f}")

[*********************100%***********************]  1 of 1 completed

                            OLS Regression Results                            
Dep. Variable:                AAPL-RF   R-squared:                       0.273
Model:                            OLS   Adj. R-squared:                  0.273
Method:                 Least Squares   F-statistic:                     1444.
Date:                Tue, 20 Oct 2026   Prob (F-statistic):               0.00
Time:                        19:17:55   Log-Likelihood:                 26882.
No. Observations:               11520   AIC:                        -5.376e+04
Df Residuals:                   11516   BIC:                        -5.373e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.000      2.708      0.0

**How to read the results**

- **Market Beta** ($\beta_{i_1}$): above 1 means AAPL moves more than the market. AAPL is more volatile than the average stock.
- **SMB Beta** ($\beta_{i_2}$): negative would mean AAPL behaves like a *large cap* stock. Here it is close to zero and not significant (`P>|t|` above 0.05), so company size explains little of AAPL's returns.
- **HML Beta** ($\beta_{i_3}$): clearly negative means AAPL behaves like a *growth* stock (low book to market ratio), not like a value stock.
- **Alpha** ($\alpha_i$): the part of the return that none of the three factors can explain. The `P>|t|` column in the summary shows whether it is statistically significant (below 0.05).
- **R-squared**: how much of AAPL's daily movement is explained by the three factors together.

$$Example\, of\, Fama\, French\, on\, the\, Velten\, Strategie\, Deutschland$$

Now let's apply the model to an actively managed fund: the *Velten Strategie Deutschland R* (ISIN DE000A2ATCU8), a German equity fund that describes itself as factor-based. Two things change compared to the AAPL example:

- **European factors instead of US factors.** A German fund should be explained by the European market, so we use the *Fama/French European 3 Factors* (daily). There are no German-only factors in the data library.
- **Currency.** Kenneth French's international factors are calculated in USD, but the fund is priced in EUR. So we first convert the fund price into USD with the EUR/USD exchange rate.

We also switch from daily to **weekly** returns. The fund's NAV, the exchange rate and the factors are not all measured at exactly the same time of day, which adds a lot of noise to daily data.

In [2]:
# Download the Fama/French European 3 factors (daily)
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Europe_3_Factors_Daily_CSV.zip"
with urllib.request.urlopen(url) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as z:
        raw = z.read(z.namelist()[0]).decode("utf-8")

header = next(i for i, line in enumerate(raw.splitlines()) if line.startswith(",Mkt-RF"))
ff_eu = pd.read_csv(io.StringIO(raw), skiprows=header, index_col=0, na_values=[-99.99]).dropna()
ff_eu.index = pd.to_datetime(ff_eu.index.astype(str).str.strip(), format="%Y%m%d")
ff_eu = ff_eu.astype(float) / 100  # percent -> decimal

# Fund price (EUR, dividend adjusted) converted into USD
prices = yf.download(["0P00019PR1.F", "EURUSD=X"], period="max", interval="1d", auto_adjust=True)["Close"].dropna()
velten_usd = prices["0P00019PR1.F"] * prices["EURUSD=X"]

data_eu = ff_eu.join(velten_usd.pct_change().rename("Velten"), how="inner").dropna()
weekly = (1 + data_eu).resample("W-FRI").prod() - 1  # compound daily returns to weekly returns
weekly["Velten-RF"] = weekly["Velten"] - weekly["RF"]

X = weekly[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)  # -> This will be your Alpha
y = weekly["Velten-RF"]

model_eu = sm.OLS(y, X).fit()
print(model_eu.summary())

alpha = model_eu.params["const"]
print("\nFama French Results (Velten Strategie Deutschland):")
print(f"Period: {weekly.index[0].date()} - {weekly.index[-1].date()}")
print(f"Alpha (weekly): {alpha:.5f}  ->  annualized: {alpha * 52:.2%}  (p-value: {model_eu.pvalues['const']:.2f})")
print(f"Market Beta: {model_eu.params['Mkt-RF']:.2f}")
print(f"SMB Beta: {model_eu.params['SMB']:.2f}")
print(f"HML Beta: {model_eu.params['HML']:.2f}")
print(f"R-squared: {model_eu.rsquared:.2f}")

[                       0%                       ]

[*********************100%***********************]  2 of 2 completed

                            OLS Regression Results                            
Dep. Variable:              Velten-RF   R-squared:                       0.774
Model:                            OLS   Adj. R-squared:                  0.773
Method:                 Least Squares   F-statistic:                     513.3
Date:                Tue, 20 Oct 2026   Prob (F-statistic):          1.16e-144
Time:                        19:17:57   Log-Likelihood:                 1260.4
No. Observations:                 453   AIC:                            -2513.
Df Residuals:                     449   BIC:                            -2496.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0001      0.001     -0.156      0.8

**What does this tell us about the fund?**

- **Market Beta ≈ 1**: the fund moves roughly one to one with the European stock market.
- **SMB Beta clearly positive**: this is the fund's biggest tilt. It holds a lot of small and mid caps, so it profits when small companies outperform large ones and suffers when they lag behind.
- **HML Beta slightly negative**: a small tilt towards growth stocks.
- **Alpha ≈ 0 and not significant**: after controlling for market, size and value, there is no outperformance left. The fund's returns can almost completely be explained by its factor exposure.
- **R-squared ≈ 0.77**: the three factors explain more than three quarters of the fund's weekly movements.

This is exactly the use case from before: the factor model shows *which risks* you are actually buying with a fund, here mainly European market risk plus a strong small cap tilt.